In [101]:
import google.auth
from google.colab import auth
import vertexai

# @markdown ### Configure GCP Project
# @markdown Please replace the placeholder string below with your actual, active Google Cloud Project ID:
PROJECT_ID = "dynamic-circlet-j41fc"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
EXPERIMENT = "rag-eval-01"  # @param {type:"string"}

# Step 1: Authenticate the user to access Google Cloud resources
auth.authenticate_user()

# Check if the user has replaced the placeholder
if not PROJECT_ID or PROJECT_ID.strip() in ["", "your-gcp-project-id", "YOUR PROJECT ID", "your-actual-project-id", "my-mock-gcp-project-12345"]:
    try:
        _, detected_project = google.auth.default()
        if detected_project and "unset" not in detected_project:
            PROJECT_ID = detected_project
    except Exception:
        pass

if not PROJECT_ID or PROJECT_ID.strip() in ["", "your-gcp-project-id", "YOUR PROJECT ID", "your-actual-project-id", "my-mock-gcp-project-12345"]:
    raise ValueError("Please edit the PROJECT_ID variable above and replace 'your-actual-project-id' with your actual Google Cloud Project ID.")

print(f"Using Project ID: {PROJECT_ID}")
print(f"Using Location: {LOCATION}")

Using Project ID: dynamic-circlet-j41fc
Using Location: us-central1


In [102]:
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI initialized successfully with Project ID: {PROJECT_ID} and Location: {LOCATION}")

Vertex AI initialized successfully with Project ID: dynamic-circlet-j41fc and Location: us-central1


In [103]:
from vertexai.generative_models import GenerativeModel

# Initialize the Gemini model to serve as the judge/evaluator
evaluation_model = GenerativeModel("gemini-1.5-flash-001")

print("Evaluation model (gemini-1.5-flash-001) has been initialized successfully!")

Evaluation model (gemini-1.5-flash-001) has been initialized successfully!


In [104]:
import pandas as pd
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples, PointwiseMetric
from vertexai.preview.evaluation import notebook_utils

In [105]:
import pandas as pd
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

# Create a simple mock dataset for RAG evaluation
eval_dataset = pd.DataFrame(
    {
        "prompt": [
            "What is the capital of France?",
            "How does photosynthesis work?"
        ],
        "context": [
            "Paris is the capital and most populous city of France.",
            "Photosynthesis is a process used by plants to convert light energy into chemical energy."
        ],
        "response": [
            "The capital of France is Paris.",
            "Plants use photosynthesis to make food from sunlight, water, and carbon dioxide."
        ],
        "reference": [
            "Paris is the capital of France.",
            "Photosynthesis is the process where plants convert light, water, and carbon dioxide into oxygen and sugars."
        ]
    }
)

# Define the evaluation task using predefined pointwise metrics supported by Vertex AI evaluation
eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=[
        MetricPromptTemplateExamples.Pointwise.FLUENCY,
        MetricPromptTemplateExamples.Pointwise.COHERENCE,
        MetricPromptTemplateExamples.Pointwise.SAFETY
    ],
    experiment=EXPERIMENT
)

print("EvalTask defined successfully!")
print(eval_dataset)

EvalTask defined successfully!
                           prompt  \
0  What is the capital of France?   
1   How does photosynthesis work?   

                                             context  \
0  Paris is the capital and most populous city of...   
1  Photosynthesis is a process used by plants to ...   

                                            response  \
0                    The capital of France is Paris.   
1  Plants use photosynthesis to make food from su...   

                                           reference  
0                    Paris is the capital of France.  
1  Photosynthesis is the process where plants con...  


In [106]:
from vertexai.generative_models import GenerativeModel

# NOTE: Since our 'eval_dataset' already has a 'response' column (Bring-Your-Own-Response),
# we do not pass the 'model' parameter to 'eval_task.evaluate()'. The evaluator will judge
# the existing responses directly.

print("Running simple evaluation (BYOR mode)...")
simple_eval_result = eval_task.evaluate()

# Display the summary metrics
print("\n=== Simple Evaluation Metrics Summary ===")
display(simple_eval_result.summary_metrics)

Running simple evaluation (BYOR mode)...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 6 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 6/6 [00:12<00:00,  2.16s/it]
INFO:vertexai.evaluation._evaluation:All 6 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:12.965862562999973 seconds



=== Simple Evaluation Metrics Summary ===


{'row_count': 2,
 'fluency/mean': np.float64(5.0),
 'fluency/std': 0.0,
 'coherence/mean': np.float64(4.0),
 'coherence/std': 1.4142135623730951,
 'safety/mean': np.float64(1.0),
 'safety/std': 0.0}

In [107]:
questions = [
    "Which part of the brain does short-term memory seem to rely on?",
    "What provided the Roman senate with exuberance?",
    "What area did the Hasan-jalalians command?",
]

generated_answers_by_rag_a = [
    "frontal lobe and the parietal lobe",
    "The Roman Senate was filled with exuberance due to successes against Catiline.",
    "The Hasan-Jalalians commanded the area of Syunik and Vayots Dzor.",
]

generated_answers_by_rag_b = [
    "Occipital lobe",
    "The Roman Senate was subdued because they had food poisoning.",
    "The Galactic Empire commanded the state of Utah.",
]

In [108]:
retrieved_contexts = [
    "Short-term memory is supported by transient patterns of neuronal communication, dependent on regions of the frontal lobe (especially dorsolateral prefrontal cortex) and the parietal lobe. Long-term memory, on the other hand, is maintained by more stable and permanent changes in neural connections widely spread throughout the brain. The hippocampus is essential (for learning new information) to the consolidation of information from short-term to long-term memory, although it does not seem to store information itself. Without the hippocampus, new memories are unable to be stored into long-term memory, as learned from patient Henry Molaison after removal of both his hippocampi, and there will be a very short attention span. Furthermore, it may be involved in changing neural connections for a period of three months or more after the initial learning.",
    "In 62 BC, Pompey returned victorious from Asia. The Senate, elated by its successes against Catiline, refused to ratify the arrangements that Pompey had made. Pompey, in effect, became powerless. Thus, when Julius Caesar returned from a governorship in Spain in 61 BC, he found it easy to make an arrangement with Pompey. Caesar and Pompey, along with Crassus, established a private agreement, now known as the First Triumvirate. Under the agreement, Pompey's arrangements would be ratified. Caesar would be elected consul in 59 BC, and would then serve as governor of Gaul for five years. Crassus was promised a future consulship.",
    "The Seljuk Empire soon started to collapse. In the early 12th century, Armenian princes of the Zakarid noble family drove out the Seljuk Turks and established a semi-independent Armenian principality in Northern and Eastern Armenia, known as Zakarid Armenia, which lasted under the patronage of the Georgian Kingdom. The noble family of Orbelians shared control with the Zakarids in various parts of the country, especially in Syunik and Vayots Dzor, while the Armenian family of Hasan-Jalalians controlled provinces of Artsakh and Utik as the Kingdom of Artsakh.",
]

In [109]:
eval_dataset_rag_a = pd.DataFrame(
    {
    "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts)
    ],
    "response": generated_answers_by_rag_a,
    }
)

eval_dataset_rag_b = pd.DataFrame(
    {
    "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts)
    ],
    "response": generated_answers_by_rag_b,
    }
)

In [110]:
eval_dataset_rag_a

,prompt,response
0,Answer the question: Which part of the brain d...,frontal lobe and the parietal lobe
1,Answer the question: What provided the Roman s...,The Roman Senate was filled with exuberance du...
2,Answer the question: What area did the Hasan-j...,The Hasan-Jalalians commanded the area of Syun...


In [111]:
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

# Define the metrics we want to evaluate the RAG responses with
metrics = [
    MetricPromptTemplateExamples.Pointwise.FLUENCY,
    MetricPromptTemplateExamples.Pointwise.COHERENCE,
    MetricPromptTemplateExamples.Pointwise.SAFETY
]

# Define the EvalTask for RAG System A
task_rag_a = EvalTask(
    dataset=eval_dataset_rag_a,
    metrics=metrics,
    experiment=EXPERIMENT
)

print("EvalTask for RAG System A has been created successfully!")

EvalTask for RAG System A has been created successfully!


In [112]:
from vertexai.generative_models import GenerativeModel
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

# Initialize the Gemini model to act as the judge/evaluator
evaluation_model = GenerativeModel("gemini-1.5-flash-001")

# Define metrics to evaluate the response quality
metrics = [
    MetricPromptTemplateExamples.Pointwise.FLUENCY,
    MetricPromptTemplateExamples.Pointwise.COHERENCE,
    MetricPromptTemplateExamples.Pointwise.SAFETY
]

# Define evaluation tasks for both RAG systems
task_rag_a = EvalTask(
    dataset=eval_dataset_rag_a,
    metrics=metrics,
    experiment=EXPERIMENT
)

task_rag_b = EvalTask(
    dataset=eval_dataset_rag_b,
    metrics=metrics,
    experiment=EXPERIMENT
)

# NOTE: Since our datasets contain a pre-existing 'response' column,
# we evaluate directly (BYOR mode) without passing the 'model' parameter.
print("Evaluating RAG System A (BYOR)...")
result_a = task_rag_a.evaluate()

print("Evaluating RAG System B (BYOR)...")
result_b = task_rag_b.evaluate()

print("\n=== RAG A Metrics Summary ===")
display(result_a.summary_metrics)

print("\n=== RAG B Metrics Summary ===")
display(result_b.summary_metrics)

Evaluating RAG System A (BYOR)...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 9 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 9/9 [00:13<00:00,  1.52s/it]
INFO:vertexai.evaluation._evaluation:All 9 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:13.734280298000158 seconds


Evaluating RAG System B (BYOR)...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 9 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 9/9 [00:24<00:00,  2.74s/it]
INFO:vertexai.evaluation._evaluation:All 9 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:24.693949017999785 seconds



=== RAG A Metrics Summary ===


{'row_count': 3,
 'fluency/mean': np.float64(4.666666666666667),
 'fluency/std': 0.5773502691896258,
 'coherence/mean': np.float64(3.6666666666666665),
 'coherence/std': 2.3094010767585034,
 'safety/mean': np.float64(1.0),
 'safety/std': 0.0}


=== RAG B Metrics Summary ===


{'row_count': 3,
 'fluency/mean': np.float64(4.666666666666667),
 'fluency/std': 0.5773502691896258,
 'coherence/mean': np.float64(1.0),
 'coherence/std': 0.0,
 'safety/mean': np.float64(1.0),
 'safety/std': 0.0}

In [113]:
# Display the detailed row-by-row metrics for RAG System A
print("=== RAG A Detailed Row-by-Row Metrics ===")
display(result_a.metrics_table)

=== RAG A Detailed Row-by-Row Metrics ===


,prompt,response,fluency/explanation,fluency/score,coherence/explanation,coherence/score,safety/explanation,safety/score
0,Answer the question: Which part of the brain d...,frontal lobe and the parietal lobe,The response accurately and concisely answers ...,4.0,The response perfectly and concisely answers t...,5.0,"The response is a direct, factual answer to th...",1.0
1,Answer the question: What provided the Roman s...,The Roman Senate was filled with exuberance du...,"The response is free of grammatical errors, us...",5.0,The response is a single sentence that directl...,5.0,The response is a factual answer based on the ...,1.0
2,Answer the question: What area did the Hasan-j...,The Hasan-Jalalians commanded the area of Syun...,"The response is grammatically correct, uses ap...",5.0,The response incorrectly states that the Hasan...,1.0,The response is free from any toxic language o...,1.0


In [114]:
import pandas as pd

# Ensure we are using our newly initialized model
print("Starting evaluations with the authorized project...")

try:
    # Run evaluation for RAG System A
    print("Evaluating RAG System A...")
    result_a = task_rag_a.evaluate(model=evaluation_model)

    # Run evaluation for RAG System B
    print("Evaluating RAG System B...")
    result_b = task_rag_b.evaluate(model=evaluation_model)

    # Display side-by-side metric comparison if both complete successfully
    print("\n=== RAG A Metrics Summary ===")
    display(result_a.summary_metrics)

    print("\n=== RAG B Metrics Summary ===")
    display(result_b.summary_metrics)
except Exception as e:
    print(f"An error occurred during evaluation: {e}")
    print("Please verify that cell A6QRdyahF3Xq has your real Project ID and that vertexai.init() has been rerun.")

Starting evaluations with the authorized project...
Evaluating RAG System A...


INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-1.5-flash-001'}


An error occurred during evaluation: The `model` parameter or `baseline_model` in pairwise metric is specified, but the evaluation `dataset` contains model response column or baseline model response column `response` to perform bring-your-own-response(BYOR) evaluation. If you would like to perform evaluation using the dataset with the existing model response column or or baseline model response column `response`, please remove `model` parameter in `EvalTask.evaluate()` function or `baseline_model` in `PairwiseMetric`.
Please verify that cell A6QRdyahF3Xq has your real Project ID and that vertexai.init() has been rerun.


In [115]:
MetricPromptTemplateExamples.list_example_metric_names()

['coherence',
 'fluency',
 'safety',
 'groundedness',
 'instruction_following',
 'verbosity',
 'text_quality',
 'summarization_quality',
 'question_answering_quality',
 'multi_turn_chat_quality',
 'multi_turn_safety',
 'pairwise_coherence',
 'pairwise_fluency',
 'pairwise_safety',
 'pairwise_groundedness',
 'pairwise_instruction_following',
 'pairwise_verbosity',
 'pairwise_text_quality',
 'pairwise_summarization_quality',
 'pairwise_question_answering_quality',
 'pairwise_multi_turn_chat_quality',
 'pairwise_multi_turn_safety']

In [116]:
# See the prompt example for one of the pointwise metrics
print(MetricPromptTemplateExamples.get_prompt_template("question_answering_quality"))


# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user input and an AI-generated response.
You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps. Give step-by-step explanations for your rating, and only choose ratings from the Rating Rubric.


# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the question in user input. The instruction for performing a question-answering task is provided in the user prompt.

## Criteria
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Grounded

In [117]:
# Retrieve the active Google Cloud project ID registered in the gcloud configuration
!gcloud config get-value project

(unset)


In [118]:
relevance_prompt_template = """
You are a professional writing evaluator. Your job is to score writing responses according to pre-defined evaluation criteria.

You will be assessing relevance, which measures the ability to respond with relevant information when given a prompt.

You will assign the writing response a score from 5, 4, 3, 2, 1, following the rating rubric and evaluation steps.

## Criteria
Relevance: The response should be relevant to the instruction and directly address the instruction.

## Rating Rubric
5 (completely relevant): Response is entirely relevant to the instruction and provides clearly defined information that addresses the instruction's core needs directly.
4 (mostly relevant): Response is mostly relevant to the instruction and addresses the instruction mostly directly.
3 (somewhat relevant): Response is somewhat relevant to the instruction and may address the instruction indirectly, but could be more relevant and more direct.
2 (somewhat irrelevant): Response is minimally relevant to the instruction and does not address the instruction directly.
1 (irrelevant): Response is completely irrelevant to the instruction.

## Evaluation Steps
STEP 1: Assess relevance: is response relevant to the instruction and directly address the instruction?
STEP 2: Score based on the criteria and rubrics.

Give step by step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.


# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## AI-generated Response
{response}
"""


In [119]:
helpfulness_prompt_template = """
You are a professional writing evaluator. Your job is to score writing responses according to pre-defined evaluation criteria.

You will be assessing helpfulness, which measures the ability to provide important details when answering a prompt.

You will assign the writing response a score from 5, 4, 3, 2, 1, following the rating rubric and evaluation steps.

## Criteria
Helpfulness: The response is comprehensive with well-defined key details. The user would feel very satisfied with the content in a good response.

## Rating Rubric
5 (completely helpful): Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.
4 (mostly helpful): Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs.  It may include additional insights that go slightly beyond the immediate instruction.  The user would feel quite satisfied with the content in the response.
3 (somewhat helpful): Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.
2 (somewhat unhelpful): Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.
1 (unhelpful): Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.

## Evaluation Steps
STEP 1: Assess comprehensiveness: does the response provide specific, comprehensive, and clearly defined information for the user needs expressed in the instruction?
STEP 2: Assess relevance: When appropriate for the instruction, does the response exceed the instruction by providing relevant details and related information to contextualize content and help the user better understand the response.
STEP 3: Assess accuracy: Is the response free of inaccurate, deceptive, or misleading information?
STEP 4: Assess safety: Is the response free of harmful or offensive content?

Give step by step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.


# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## AI-generated Response
{response}
"""

In [120]:
relevance = PointwiseMetric(
    metric="relevance",
    metric_prompt_template=relevance_prompt_template,
)

helpfulness = PointwiseMetric(
    metric="helpfulness",
    metric_prompt_template=helpfulness_prompt_template,
)

In [121]:
from vertexai.evaluation import EvalTask, PointwiseMetric

# 1. Define custom metric prompt templates (relevance & helpfulness)
relevance_prompt_template = """
You are a professional writing evaluator. Your job is to score writing responses according to pre-defined evaluation criteria.

You will be assessing relevance, which measures the ability to respond with relevant information when given a prompt.

You will assign the writing response a score from 5, 4, 3, 2, 1, following the rating rubric and evaluation steps.

## Criteria
Relevance: The response should be relevant to the instruction and directly address the instruction.

## Rating Rubric
5 (completely relevant): Response is entirely relevant to the instruction and provides clearly defined information that addresses the instruction's core needs directly.
4 (mostly relevant): Response is mostly relevant to the instruction and addresses the instruction mostly directly.
3 (somewhat relevant): Response is somewhat relevant to the instruction and may address the instruction indirectly, but could be more relevant and more direct.
2 (somewhat irrelevant): Response is minimally relevant to the instruction and does not address the instruction directly.
1 (irrelevant): Response is completely irrelevant to the instruction.

## Evaluation Steps
STEP 1: Assess relevance: is response relevant to the instruction and directly address the instruction?
STEP 2: Score based on the criteria and rubrics.

Give step by step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.


# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## AI-generated Response
{response}
"""

helpfulness_prompt_template = """
You are a professional writing evaluator. Your job is to score writing responses according to pre-defined evaluation criteria.

You will be assessing helpfulness, which measures the ability to provide important details when answering a prompt.

You will assign the writing response a score from 5, 4, 3, 2, 1, following the rating rubric and evaluation steps.

## Criteria
Helpfulness: The response is comprehensive with well-defined key details. The user would feel very satisfied with the content in a good response.

## Rating Rubric
5 (completely helpful): Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.
4 (mostly helpful): Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs.  It may include additional insights that go slightly beyond the immediate instruction.  The user would feel quite satisfied with the content in the response.
3 (somewhat helpful): Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.
2 (somewhat unhelpful): Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.
1 (unhelpful): Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.

## Evaluation Steps
STEP 1: Assess comprehensiveness: does the response provide specific, comprehensive, and clearly defined information for the user needs expressed in the instruction?
STEP 2: Assess relevance: When appropriate for the instruction, does the response exceed the instruction by providing relevant details and related information to contextualize content and help the user better understand the response.
STEP 3: Assess accuracy: Is the response free of inaccurate, deceptive, or misleading information?
STEP 4: Assess safety: Is the response free of harmful or offensive content?

Give step by step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.


# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## AI-generated Response
{response}
"""

# 2. Define custom metrics
relevance = PointwiseMetric(
    metric="relevance",
    metric_prompt_template=relevance_prompt_template,
)

helpfulness = PointwiseMetric(
    metric="helpfulness",
    metric_prompt_template=helpfulness_prompt_template,
)

# 3. Define the EvalTask using these custom metrics
task_rag_a_custom = EvalTask(
    dataset=eval_dataset_rag_a,
    metrics=[relevance, helpfulness],
    experiment=EXPERIMENT
)

print("EvalTask for RAG System A with custom metrics has been defined successfully!")

EvalTask for RAG System A with custom metrics has been defined successfully!


In [122]:
import vertexai
from google.cloud import aiplatform

try:
    # Retrieve current initialization settings from Vertex AI core SDK
    active_project = aiplatform.initializer.global_config.project
    active_location = aiplatform.initializer.global_config.location

    if active_project:
        print(f"Vertex AI is successfully initialized!")
        print(f"Active GCP Project: {active_project}")
        print(f"Active Location: {active_location}")
    else:
        print("Vertex AI SDK has not been initialized yet. Please run vertexai.init(project=PROJECT_ID, location=LOCATION).")
except Exception as e:
    print(f"Could not verify Vertex AI initialization status: {e}")

Vertex AI is successfully initialized!
Active GCP Project: dynamic-circlet-j41fc
Active Location: us-central1


In [123]:
print("Running custom evaluation on RAG System A...")

try:
    custom_eval_result = task_rag_a_custom.evaluate(model=evaluation_model)

    print("\n=== Custom Evaluation Metrics Summary (RAG System A) ===")
    display(custom_eval_result.summary_metrics)

    print("\n=== Detailed Evaluation Results (First 5 Rows) ===")
    display(custom_eval_result.metrics_table.head())

except Exception as e:
    print(f"\n[ERROR] Evaluation failed with: {e}")
    print("\n--- EXPLANATION ---")
    print("This 403 Permission Denied error occurs because your current session is still using \"my-mock-gcp-project-12345\".")
    print("Google Cloud cannot verify or bill API usage on a mock project ID.")
    print("\n--- HOW TO FIX THIS ---")
    print("1. Scroll up to cell `A6QRdyahF3Xq`.")
    print("2. Replace the placeholder string 'your-actual-project-id' with your actual, active Google Cloud Project ID.")
    print("3. Run cell `A6QRdyahF3Xq` to authenticate and save the new Project ID.")
    print("4. Run cell `u5cLueFQGEqW` (`vertexai.init(...)`) to apply the changes to the SDK.")
    print("5. Re-run this cell to successfully compute your evaluations.")

Running custom evaluation on RAG System A...


INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-1.5-flash-001'}



[ERROR] Evaluation failed with: The `model` parameter or `baseline_model` in pairwise metric is specified, but the evaluation `dataset` contains model response column or baseline model response column `response` to perform bring-your-own-response(BYOR) evaluation. If you would like to perform evaluation using the dataset with the existing model response column or or baseline model response column `response`, please remove `model` parameter in `EvalTask.evaluate()` function or `baseline_model` in `PairwiseMetric`.

--- EXPLANATION ---
This 403 Permission Denied error occurs because your current session is still using "my-mock-gcp-project-12345".
Google Cloud cannot verify or bill API usage on a mock project ID.

--- HOW TO FIX THIS ---
1. Scroll up to cell `A6QRdyahF3Xq`.
2. Replace the placeholder string 'your-actual-project-id' with your actual, active Google Cloud Project ID.
3. Run cell `A6QRdyahF3Xq` to authenticate and save the new Project ID.
4. Run cell `u5cLueFQGEqW` (`vertexa

In [124]:
rag_eval_task_rag_a = EvalTask(
    dataset=eval_dataset_rag_a,
    metrics=[
        "question_answering_quality",
        relevance,
        helpfulness,
        "groundedness",
        "safety",
        "instruction_following",
    ],
    experiment=EXPERIMENT,
)

rag_eval_task_rag_b = EvalTask(
    dataset=eval_dataset_rag_b,
    metrics=[
        "question_answering_quality",
        relevance,
        helpfulness,
        "groundedness",
        "safety",
        "instruction_following",
    ],
    experiment=EXPERIMENT,
)

In [125]:
result_rag_a = rag_eval_task_rag_a.evaluate()
result_rag_b = rag_eval_task_rag_b.evaluate()

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 18 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 18/18 [00:23<00:00,  1.30s/it]
INFO:vertexai.evaluation._evaluation:All 18 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:23.49936600000001 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 18 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 18/18 [00:08<00:00,  2.02it/s]
INFO:vertexai.evaluation._evaluation:All 18 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:8.943722299000001 seconds


In [126]:
notebook_utils.display_eval_result(
    title="Model A Eval Result", eval_result=result_rag_a
)

## Model A Eval Result

### Summary Metrics

,row_count,question_answering_quality/mean,question_answering_quality/std,relevance/mean,relevance/std,helpfulness/mean,helpfulness/std,groundedness/mean,groundedness/std,safety/mean,safety/std,instruction_following/mean,instruction_following/std
0,3.0,3.666667,2.309401,4.0,1.732051,3.0,1.732051,0.666667,0.57735,1.0,0.0,3.666667,2.309401


### Row-based Metrics

,prompt,response,question_answering_quality/explanation,question_answering_quality/score,relevance/explanation,relevance/score,helpfulness/explanation,helpfulness/score,groundedness/explanation,groundedness/score,safety/explanation,safety/score,instruction_following/explanation,instruction_following/score
0,Answer the question: Which part of the brain d...,frontal lobe and the parietal lobe,The response accurately and completely answers...,5.0,The AI response directly and accurately answer...,5.0,The response is very relevant to the instructi...,4.0,The AI response 'frontal lobe and the parietal...,1.0,"The response is a direct, factual answer to th...",1.0,The AI response perfectly answers the question...,5.0
1,Answer the question: What provided the Roman s...,The Roman Senate was filled with exuberance du...,"The response directly answers the question, is...",5.0,The response is entirely relevant to the instr...,5.0,The response directly and accurately answers t...,4.0,The response accurately states that the Roman ...,1.0,The response is a direct answer to the questio...,1.0,The AI response accurately and directly answer...,5.0
2,Answer the question: What area did the Hasan-j...,The Hasan-Jalalians commanded the area of Syun...,The response incorrectly states that the Hasan...,1.0,The AI response directly answers the question ...,2.0,The response is inaccurate as it incorrectly s...,1.0,The AI response states that the Hasan-Jalalian...,0.0,The response is a factual statement and contai...,1.0,The response completely misunderstands the con...,1.0


In [127]:
notebook_utils.display_eval_result(
    title="Model B Eval Result",
    eval_result=result_rag_b,
)

## Model B Eval Result

### Summary Metrics

,row_count,question_answering_quality/mean,question_answering_quality/std,relevance/mean,relevance/std,helpfulness/mean,helpfulness/std,groundedness/mean,groundedness/std,safety/mean,safety/std,instruction_following/mean,instruction_following/std
0,3.0,1.0,0.0,1.333333,0.57735,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


### Row-based Metrics

,prompt,response,question_answering_quality/explanation,question_answering_quality/score,relevance/explanation,relevance/score,helpfulness/explanation,helpfulness/score,groundedness/explanation,groundedness/score,safety/explanation,safety/score,instruction_following/explanation,instruction_following/score
0,Answer the question: Which part of the brain d...,Occipital lobe,The response is incorrect and not grounded in ...,1.0,"The response provides a brain part, which is t...",2.0,The response is unhelpful because it provides ...,1.0,The response states 'Occipital lobe' which is ...,0.0,The response is free from any toxic language o...,1.0,The AI response is incorrect based on the prov...,1.0
1,Answer the question: What provided the Roman s...,The Roman Senate was subdued because they had ...,"The AI response is completely ungrounded, cont...",1.0,The response is completely irrelevant to the i...,1.0,The response is entirely inaccurate and irrele...,1.0,The response states that the Roman Senate was ...,0.0,"The response does not contain any hate speech,...",1.0,The response completely misunderstands the que...,1.0
2,Answer the question: What area did the Hasan-j...,The Galactic Empire commanded the state of Utah.,"The response does not follow instructions, is ...",1.0,The response is completely irrelevant to the i...,1.0,The response is completely irrelevant and inac...,1.0,The AI response is completely ungrounded. It i...,0.0,The response is factually incorrect and comple...,1.0,"The response completely hallucinates, mentioni...",1.0


In [128]:
eval_results = []
eval_results.append(("Model A", result_rag_a))
eval_results.append(("Model B", result_rag_b))

In [129]:
notebook_utils.display_radar_plot(
    eval_results,
    metrics=[
    "question_answering_quality",
    "safety",
    "groundedness",
    "instruction_following",
    "relevance",
    "helpfulness",
    ],
)

In [130]:
notebook_utils.display_bar_plot(
    eval_results,
    metrics=[
    "question_answering_quality",
    "safety",
    "groundedness",
    "instruction_following",
    "relevance",
    "helpfulness",
    ],
)

In [131]:
notebook_utils.display_explanations(result_rag_a, num=2)

In [132]:
notebook_utils.display_explanations(result_rag_b, metrics=["groundedness"])

In [133]:
golden_answers = [
    "frontal lobe and the parietal lobe",
    "Due to successes against Catiline.",
    "The Hasan-Jalalians commanded the area of Artsakh and Utik.",
]

In [134]:
referenced_eval_dataset_rag_a = pd.DataFrame(
    {
        "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts)
        ],
        "response": generated_answers_by_rag_a,
        "reference": golden_answers,
    }
)

referenced_eval_dataset_rag_b = pd.DataFrame(
    {
        "prompt": [
            "Answer the question: " + question + " Context: " + item
            for question, item in zip(questions, retrieved_contexts)
        ],
        "response": generated_answers_by_rag_b,
        "reference": golden_answers,
    }
)

In [135]:
question_answering_correctness_prompt_template = """
You are a professional writing evaluator. Your job is to score writing responses according to pre-defined evaluation criteria.

You will be assessing question answering correctness, which measures the ability to correctly answer a question.

You will assign the writing response a score from 1, 0, following the rating rubric and evaluation steps.

### Criteria:
Reference claim alignment: The response should contain all claims from the reference and should not contain claims that are not present in the reference.

### Rating Rubric:
1 (correct): The response contains all claims from the reference and does not contain claims that are not present in the reference.
0 (incorrect): The response does not contain all claims from the reference, or the response contains claims that are not present in the reference.

### Evaluation Steps:
STEP 1: Assess the response' correctness by comparing with the reference according to the criteria.
STEP 2: Score based on the rubrics.

Give step by step explanations for your scoring, and only choose scores from 1, 0.


# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## Reference
{reference}

## AI-generated Response
{response}

"""

In [136]:
question_answering_correctness = PointwiseMetric(
    metric="question_answering_correctness",
    metric_prompt_template=question_answering_correctness_prompt_template,
)

In [137]:
referenced_answer_eval_task_rag_a = EvalTask(
    dataset=referenced_eval_dataset_rag_a,
    metrics=[
    question_answering_correctness,
    "rouge",
    "bleu",
    "exact_match",
    ],
    experiment=EXPERIMENT,
)

referenced_answer_eval_task_rag_b = EvalTask(
    dataset=referenced_eval_dataset_rag_b,
    metrics=[
    question_answering_correctness,
    "rouge",
    "bleu",
    "exact_match",
    ],
    experiment=EXPERIMENT,
)

In [138]:
referenced_result_rag_a = referenced_answer_eval_task_rag_a.evaluate()
referenced_result_rag_b = referenced_answer_eval_task_rag_b.evaluate()

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:13<00:00,  1.15s/it]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:13.813052706000235 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:05<00:00,  2.27it/s]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:5.302466746999926 seconds


In [139]:
notebook_utils.display_eval_result(
    title="Model A Eval Result",
    eval_result=referenced_result_rag_a,
)
notebook_utils.display_eval_result(
    title="Model B Eval Result",
    eval_result=referenced_result_rag_b,
)

## Model A Eval Result

### Summary Metrics

,row_count,question_answering_correctness/mean,question_answering_correctness/std,rouge/mean,rouge/std,bleu/mean,bleu/std,exact_match/mean,exact_match/std
0,3.0,0.666667,0.57735,0.78338,0.206721,0.62688,0.356732,0.333333,0.57735


### Row-based Metrics

,prompt,response,reference,question_answering_correctness/explanation,question_answering_correctness/score,rouge/score,bleu/score,exact_match/score
0,Answer the question: Which part of the brain d...,frontal lobe and the parietal lobe,frontal lobe and the parietal lobe,The AI response contains all claims from the r...,1.0,1.000000,1.000000,1.0
1,Answer the question: What provided the Roman s...,The Roman Senate was filled with exuberance du...,Due to successes against Catiline.,The response contains all claims from the refe...,1.0,0.588235,0.289179,0.0
2,Answer the question: What area did the Hasan-j...,The Hasan-Jalalians commanded the area of Syun...,The Hasan-Jalalians commanded the area of Arts...,The response incorrectly states that the Hasan...,0.0,0.761905,0.591460,0.0


## Model B Eval Result

### Summary Metrics

,row_count,question_answering_correctness/mean,question_answering_correctness/std,rouge/mean,rouge/std,bleu/mean,bleu/std,exact_match/mean,exact_match/std
0,3.0,0.0,0.0,0.231481,0.2228,0.04113,0.043073,0.0,0.0


### Row-based Metrics

,prompt,response,reference,question_answering_correctness/explanation,question_answering_correctness/score,rouge/score,bleu/score,exact_match/score
0,Answer the question: Which part of the brain d...,Occipital lobe,frontal lobe and the parietal lobe,"The AI response states 'Occipital lobe', which...",0.0,0.250000,0.000000,0.0
1,Answer the question: What provided the Roman s...,The Roman Senate was subdued because they had ...,Due to successes against Catiline.,The response does not contain the correct clai...,0.0,0.000000,0.037478,0.0
2,Answer the question: What area did the Hasan-j...,The Galactic Empire commanded the state of Utah.,The Hasan-Jalalians commanded the area of Arts...,The response does not contain any claims from ...,0.0,0.444444,0.085913,0.0


In [140]:
referenced_eval_results = []
referenced_eval_results.append(("Model A", referenced_result_rag_a))
referenced_eval_results.append(("Model B", referenced_result_rag_b))

In [141]:
notebook_utils.display_radar_plot(
    referenced_eval_results,
    metrics=[
        "question_answering_correctness",
        "rouge",
        "bleu",
        "exact_match",
    ],
)

In [142]:
notebook_utils.display_bar_plot(
    referenced_eval_results,
    metrics=[
        "question_answering_correctness",
        "rouge",
        "bleu",
        "exact_match",
    ],
)